# Formal corpus freeze and replay

This notebook binds the local analysis to `formal-corpus-release-20261117-v1`. It
checks the full formal roster, the derived document handoff and the recorded source-use
exclusions, then replays the document cohort selection. Run in a fresh project kernel.

Local output includes a verification receipt, a native-body canonical inventory covering
eligible and structurally held documents, and a reference-only metadata ledger
for the historical source-use holds. Native text, the formal release,
evaluation splits and chunk eligibility remain under their original contracts.

In [ ]:
import hashlib
import json
import tempfile
from pathlib import Path

import nbformat
from pyspark import StorageLevel
from pyspark.sql import functions as F

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "notebooks/13_document_eda_and_unsupervised_preparation.ipynb").is_file()
)
VERSION = "formal-corpus-release-20261117-v1"
SPLITS = {"train": 646_414, "validation": 36_110, "test": 35_806}
POINTER = ROOT / "data/processed/corpus_releases/CURRENT_RELEASE.json"
HISTORICAL = ROOT / "work_packages/prechunk_pilot_20261113/v2/POOL_FORMAL_METADATA.parquet"
USAGE = ROOT / (
    "data/processed/preprocessing/platform_readmission_v3_topic_20261113/"
    "KNOWN_USAGE_HOLDS.parquet"
)
USAGE_PROOFS = ROOT / (
    "docs/research/2026-11-13-preprocessing-eligibility-audit/SAVED_USAGE_SIGNAL_PROOFS.json"
)

def sha256_file(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def split_counts(frame, split):
    return {row[split]: row["count"] for row in frame.groupBy(split).count().collect()}

def require(condition, message):
    if not condition:
        raise ValueError(message)

pointer = read_json(POINTER)
receipt_path = ROOT / pointer["receipt"]
catalog_path = ROOT / pointer["catalog"]
release = receipt_path.parent
membership_path = release / "RELEASE_MEMBERSHIP.parquet"
passage_path = release / "LONG_PASSAGE_FINGERPRINTS.parquet"
receipt = read_json(receipt_path)
pinned_hashes = {
    POINTER: "10ffb03cd04c859c28a6bebe0466ad322996e5ab9caed332b287d3bc51b5d1b8",
    membership_path: "abe5fbca4d1a0596c03e351cd38232fb60e041c28277c83cc6e802766d8bd5ae",
    catalog_path: receipt["catalog_sha256"],
    passage_path: receipt["passage_fingerprint_sha256"],
    HISTORICAL: "602eca9e84478b83f94bc5e675d5d603d4a955d924b862b138f9ae17eafdaca1",
    USAGE: "9d49cc6a62c878b38b5f107df78b409859fef674fcf4d84fedd7c9f316d195cd",
    USAGE_PROOFS: "bf707bbddfeaa5609c4b098828b0d1a5d6878f970fdd591534e93e2221aa17be",
}
require(
    pointer["version"] == receipt["version"] == VERSION
    and pointer["formal_records"] == receipt["formal_records"] == sum(SPLITS.values())
    and receipt["membership_sha256"] == pinned_hashes[membership_path],
    "The active pointer does not resolve to the frozen formal release.",
)
for source, digest in pinned_hashes.items():
    require(sha256_file(source) == digest, f"Input checksum mismatch: {source.name}")

part_manifests = [
    release / "BASELINE_MATERIALIZED.json",
    release / "SUPPLEMENT_MATERIALIZED.json",
]
release_parts = [part for manifest in part_manifests for part in read_json(manifest)["parts"]]
part_paths = [ROOT / part["path"] for part in release_parts]
require(
    len(part_paths) == len(set(part_paths)) == 150
    and set(part_paths) == set((release / "records").glob("*.parquet")),
    "The frozen release part inventory does not reconcile.",
)
for part, path in zip(release_parts, part_paths, strict=True):
    require(sha256_file(path) == part["sha256"], f"Release part checksum mismatch: {path.name}")

# Notebook 13 verifies the complete Notebook 05 input and loads the document hold tables.
eda_path = ROOT / "notebooks/13_document_eda_and_unsupervised_preparation.ipynb"
eda_sources = {cell.id: cell.source for cell in nbformat.read(eda_path, as_version=4).cells}
eda_scope = {"__name__": "__main__"}

def replay_eda_cell(cell_id):
    exec(compile(eda_sources[cell_id], f"{eda_path.name}:{cell_id}", "exec"), eda_scope)

replay_eda_cell("13-bind")
spark = eda_scope["spark"]
parents = eda_scope["parents"]
print(f"Bound {VERSION} and {len(part_paths)} release parts.")

In [ ]:
membership = spark.read.parquet(str(membership_path)).persist(StorageLevel.MEMORY_AND_DISK)
formal_keys = membership.where(F.col("formal_eligible")).select(
    "release_row", "release_group_id", "ml_split"
)
require(membership.count() == 723_158, "Unexpected materialized membership count.")
require(
    membership.select("release_row").distinct().count() == 723_158,
    "The materialized roster contains repeated release rows.",
)
formal_counts = split_counts(formal_keys, "ml_split")
require(formal_counts == SPLITS, f"Frozen split mismatch: {formal_counts}")

release_records = spark.read.parquet(*map(str, part_paths)).select(
    "release_row", "native_identity", "source_host", "body_sha256"
).persist(StorageLevel.MEMORY_AND_DISK)
require(
    release_records.count() == release_records.select("release_row").distinct().count()
    == 723_158,
    "The materialized record inventory is incomplete or duplicated.",
)
keys = ["release_row", "release_group_id", "ml_split", "native_identity"]
formal_roster = formal_keys.join(release_records, "release_row")
handoff_roster = parents.select(*keys)
roster_checks = {
    "formal_rows": formal_roster.count(),
    "distinct_native_identities": formal_roster.select("native_identity").distinct().count(),
    "release_minus_handoff": formal_roster.select(*keys).exceptAll(handoff_roster).count(),
    "handoff_minus_release": handoff_roster.exceptAll(formal_roster.select(*keys)).count(),
    "handoff_body_hash_or_length_failures": parents.where(
        ~F.sha2("clean_text", 256).eqNullSafe(F.col("clean_sha256"))
        | ~F.length("clean_text").eqNullSafe(F.col("clean_chars"))
        | F.col("clean_text").isNull()
        | F.col("clean_sha256").isNull()
    ).count(),
}
require(
    roster_checks["formal_rows"] == roster_checks["distinct_native_identities"]
    == sum(SPLITS.values()) and all(
        roster_checks[key] == 0 for key in (
            "release_minus_handoff", "handoff_minus_release",
            "handoff_body_hash_or_length_failures",
        )
    ),
    f"Full release replay failed: {roster_checks}",
)
print(json.dumps({"formal_splits": formal_counts, **roster_checks}, indent=2))

# Keep structural holds in the retained inventory, using exact native body hashes.
# Cleaned-empty or otherwise convergent derived strings can represent different originals.
retained = release_records.join(membership.select("release_row", "formal_eligible"), "release_row")
require(
    retained.count() == 723_158
    and not retained.where(
        F.col("body_sha256").isNull() | F.col("formal_eligible").isNull()
    ).limit(1).count(),
    "A retained record lacks an original body hash or disposition.",
)
canonical_inventory = (
    retained.select(
        "body_sha256",
        F.struct(
            (~F.col("formal_eligible")).alias("reference_priority"),
            "release_row",
            "formal_eligible",
        ).alias("candidate"),
    )
    .groupBy("body_sha256").agg(F.min("candidate").alias("canonical"))
    .select("body_sha256", "canonical.release_row", "canonical.formal_eligible")
    .persist(StorageLevel.MEMORY_AND_DISK)
)
retained_counts = {
    row.formal_eligible: row["count"]
    for row in retained.groupBy("formal_eligible").count().collect()
}
canonical_counts = {
    row.formal_eligible: row["count"]
    for row in canonical_inventory.groupBy("formal_eligible").count().collect()
}
require(
    retained_counts == {True: 718_330, False: 4_828}
    and canonical_counts == {True: 717_334, False: 4_826},
    "The retained and canonical database populations did not reconcile.",
)
database_inventory = {
    "retained_records": sum(retained_counts.values()),
    "unique_native_body_hashes": sum(canonical_counts.values()),
    "formal_records": retained_counts[True],
    "unique_formal_native_body_hashes": canonical_counts[True],
    "structural_hold_records": retained_counts[False],
    "additional_unique_structural_hold_native_bodies": canonical_counts[False],
}
print(json.dumps(database_inventory, indent=2))


In [ ]:
historical = spark.read.parquet(str(HISTORICAL)).select(
    "native_identity", "main_split", "source_id", "url_source", "body_sha256"
)
history_counts = split_counts(historical, "main_split")
require(
    history_counts == {"train": 721_793, "validation": 39_990, "test": 39_776},
    "The historical partition does not match its verified counts.",
)
usage_ids = spark.read.parquet(str(USAGE)).select("native_identity")
evidence = [
    (
        proof["host"], proof["metadata"]["response_sha256"],
        " | ".join(proof["signal_lines"]),
    )
    for proof in read_json(USAGE_PROOFS)
]
evidence_by_host = spark.createDataFrame(
    evidence, "source_host string, evidence_response_sha256 string, recorded_signal string"
)
archive = (
    historical.join(F.broadcast(usage_ids), "native_identity")
    .withColumn(
        "source_host",
        F.regexp_replace(F.lower(F.expr("try_parse_url(url_source, 'HOST')")), r"^www[.]", ""),
    )
    .drop("url_source")
    .join(F.broadcast(evidence_by_host), "source_host", "left")
    .withColumnRenamed("main_split", "historical_split")
    .withColumn("disposition", F.lit("reference_only_recorded_source_use_hold"))
    .withColumn("eligible_for_fit", F.lit(False))
    .persist(StorageLevel.MEMORY_AND_DISK)
)
archive_counts = split_counts(archive, "historical_split")
source_use_checks = {
    "known_identity_rows": usage_ids.count(),
    "known_distinct_identities": usage_ids.distinct().count(),
    "archive_distinct_identities": archive.select("native_identity").distinct().count(),
    "matched_by_identity_in_materialized_release": release_records.join(
        F.broadcast(usage_ids), "native_identity", "left_semi"
    ).count(),
    "matched_by_evidence_host_in_formal_release": formal_roster.where(
        F.regexp_replace(F.lower("source_host"), r"^www[.]", "").isin(
            [item[0] for item in evidence]
        )
    ).count(),
    "archive_rows_with_matching_host_evidence": archive.where(
        F.col("evidence_response_sha256").isNotNull()
    ).count(),
}
require(
    archive_counts == {"train": 60_456, "validation": 3_361, "test": 3_405}
    and all(
        source_use_checks[key] == 67_222
        for key in ("known_identity_rows", "known_distinct_identities", "archive_distinct_identities")
    )
    and source_use_checks["matched_by_identity_in_materialized_release"] == 0
    and source_use_checks["matched_by_evidence_host_in_formal_release"] == 0,
    f"The recorded source-use exclusions did not reconcile: {source_use_checks}",
)
print(json.dumps({"historical_reference_holds": archive_counts, **source_use_checks}, indent=2))

In [ ]:
eda_manifest_path = eda_scope["output"] / "MANIFEST.json"
eda_manifest = read_json(eda_manifest_path)
require(eda_manifest["identity"] == eda_scope["identity"], "Notebook 13 input binding changed.")
# The cohort cell uses the metadata projection defined at the start of the census cell.
census_source = eda_sources["13-census"]
projection, separator, _ = census_source.partition("\ncensus =")
require(bool(separator), "Notebook 13 census cell needs a separate metadata projection.")
exec(compile(projection, f"{eda_path.name}:13-census metadata", "exec"), eda_scope)
replay_eda_cell("13-cohort")
cohort_columns = ["release_row", "clean_sha256"]
selected = eda_scope["selection"].select(*cohort_columns)
saved = spark.read.parquet(str(eda_scope["output"] / "train_cohort")).select(*cohort_columns)
verification = {
    "canonical_train_documents": eda_scope["population"],
    "selected_train_documents": selected.count(),
    "selected_minus_saved": selected.exceptAll(saved).count(),
    "saved_minus_selected": saved.exceptAll(selected).count(),
}
require(
    verification == {
        "canonical_train_documents": 641_820, "selected_train_documents": 20_000,
        "selected_minus_saved": 0, "saved_minus_selected": 0,
    },
    f"Replayed cohort does not match the frozen document sample: {verification}",
)
print(json.dumps(verification, indent=2))

In [ ]:
freeze_base = ROOT / ".cache/corpus-freeze"
freeze_base.mkdir(mode=0o700, exist_ok=True)
identity = {
    "version": VERSION,
    "source_hashes": {
        source.relative_to(ROOT).as_posix(): sha256_file(source)
        for source in [
            *pinned_hashes, receipt_path, *part_manifests, eda_manifest_path,
            eda_scope["handoff"]["handoff_manifest_path"], eda_scope["qa_manifest_path"],
        ]
    },
}
notebook = nbformat.read(ROOT / "notebooks/17_corpus_freeze_replay.ipynb", as_version=4)
identity["notebook_sources_sha256"] = eda_scope["canonical_sha256"]([
    (cell.id, cell.source) for cell in notebook.cells
])
key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:16]
output = freeze_base / f"{VERSION}-{key}"
measurement = {
    "identity": identity,
    "formal_splits": formal_counts,
    "formal_total": sum(formal_counts.values()),
    "formal_materialized": 723_158,
    "structural_holds": 4_828,
    "release_parts_verified": len(part_paths),
    "release_replay": roster_checks,
    "database_inventory": database_inventory,
    "historical_splits": history_counts,
    "historic_source_use_hold_splits": archive_counts,
    "source_use_checks": source_use_checks,
    "document_cohort_replay": verification,
    "archive_kind": "private_metadata_index_with_original_metadata_lookup",
    "chunk_model_input_approved": False,
}

def inventory_for(directory, table):
    return {
        path.relative_to(directory).as_posix(): sha256_file(path)
        for path in sorted((directory / table).glob("*.parquet"))
    }

if output.exists():
    previous = read_json(output / "MANIFEST.json")
    files = {
        table: inventory_for(output, table)
        for table in ("reference_usage_holds", "canonical_release_inventory")
    }
    require(
        all(files.values()) and previous == {**measurement, "table_parts_sha256": files},
        "The saved freeze differs from the replay.",
    )
else:
    with tempfile.TemporaryDirectory(prefix=".stage-", dir=freeze_base) as temp:
        stage = Path(temp) / "snapshot"
        stage.mkdir(mode=0o700)
        tables = {"reference_usage_holds": archive, "canonical_release_inventory": canonical_inventory}
        for table, frame in tables.items():
            frame.coalesce(2).write.mode("errorifexists").option("compression", "zstd").parquet(
                str(stage / table)
            )
        files = {table: inventory_for(stage, table) for table in tables}
        require(all(files.values()), "A verification table has no data files.")
        (stage / "MANIFEST.json").write_text(
            json.dumps({**measurement, "table_parts_sha256": files}, indent=2, sort_keys=True)
            + "\n",
            encoding="utf-8",
        )
        stage.rename(output)

spark.stop()
print(f"Verified corpus freeze: {output.relative_to(ROOT) / 'MANIFEST.json'}")